# Step 5D.2c — UNSW forensic diagnostic (validation AP trajectory, categorical contract, execution mode)

Мета: НЕ перезапускати весь Step 5D.2, НЕ міняти модель чи split, а локалізувати, ЧОМУ
незалежна reproduction UNSW-F0/F1 (Step 5D.2b, розділ 6) дає інший `best_iteration`
(277 замість 33 для F0, 249 замість 290 для F1), хоча feature contract і temporal split
підтверджено ідентичними, а 5 повторів у поточному середовищі дають бітово однаковий
результат (Step 5D.2b, розділ 7c).

Цей notebook НЕ обмежується підсумковим test AP -- він захоплює ПОВНУ validation AP
trajectory (по кожній boosting-ітерації), явно перевіряє categorical dtype/codes/order,
і документує execution mode (row-wise/col-wise, кількість потоків). Оригінальна (Step
5D.2) trajectory НЕ відновлюється -- вона ніколи не зберігалась (ні model object, ні
`evals_result_`), і це чесно констатується нижче, а не підмінюється вигаданими числами.

Область застосування: ЛИШЕ UNSW-F0/F1. CIC вже підтверджено бітово точно (Step 5D.2b,
розділ 7) і тут не диагностується повторно.

In [1]:
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5d2_audit" / "unsw_forensic"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())
print("AUDIT_OUT_DIR:", AUDIT_OUT_DIR)


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True
AUDIT_OUT_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_forensic


## 1. Середовище виконання -- версії, потоки, execution mode

Записується ЯВНО, тому що це головна відсутня ланка: Step 5D.2 (оригінальний прогін) НЕ
зберігав цю інформацію. Порівняти "оригінал vs поточне" на рівні версій бібліотек чи
кількості потоків наразі неможливо -- можливо лише задокументувати поточне середовище й
шукати історичний запис (conda env / pip freeze / git-лог) окремо, поза цим notebook.

In [2]:
import sys
import lightgbm as lgb
import sklearn

env_info = dict(
    python=sys.version.split()[0],
    lightgbm=lgb.__version__,
    numpy=np.__version__,
    pandas=pd.__version__,
    sklearn=sklearn.__version__,
)
print("Версії середовища виконання (ПОТОЧНОГО прогону):")
for k, v in env_info.items():
    print(f"  {k}: {v}")

n_cpu = os.cpu_count()
omp_threads = os.environ.get("OMP_NUM_THREADS")
lgb_threads_env = os.environ.get("LIGHTGBM_NUM_THREADS") or os.environ.get("LGB_NUM_THREADS")
print(f"\nos.cpu_count()={n_cpu}  OMP_NUM_THREADS={omp_threads!r}  "
      f"LIGHTGBM/LGB_NUM_THREADS env={lgb_threads_env!r}")
print(
    "\n⚠ LightGBM за замовчуванням (без явного num_threads/force_row_wise/force_col_wise у "
    "LGB_PARAMS) сам обирає row-wise чи col-wise виконання ЧЕРЕЗ RUNTIME-БЕНЧМАРК під час "
    "побудови Dataset (лог 'Auto-choosing row-wise multi-threading, the overhead of testing "
    "was X seconds'). Це рішення залежить від виміряного wall-clock часу мікро-тесту в "
    "конкретний момент запуску -- тобто воно НЕ контролюється seed і теоретично може "
    "відрізнятись між двома прогонами на тій самій машині з тими самими бібліотеками, якщо "
    "момент запуску відрізняється (завантаження системи, кеш, throttling). Це -- поки що "
    "НЕ доведена, а лише правдоподібна кандидат-причина, яку варто тримати в полі зору при "
    "інтерпретації розділу 5 нижче."
)

print(
    "\n❗ Версії бібліотек / кількість потоків / execution mode ОРИГІНАЛЬНОГО прогону Step "
    "5D.2 НЕ відновлені тут і не можуть бути відновлені з наявних артефактів (збережено лише "
    "агреговані метрики). Якщо історичний conda env / pip freeze / git-лог існує окремо -- "
    "це відповість на питання точніше, ніж будь-яка діагностика в цьому notebook."
)


Версії середовища виконання (ПОТОЧНОГО прогону):
  python: 3.11.15
  lightgbm: 4.7.0
  numpy: 2.4.4
  pandas: 3.0.5
  sklearn: 1.9.0

os.cpu_count()=32  OMP_NUM_THREADS=None  LIGHTGBM/LGB_NUM_THREADS env=None

⚠ LightGBM за замовчуванням (без явного num_threads/force_row_wise/force_col_wise у LGB_PARAMS) сам обирає row-wise чи col-wise виконання ЧЕРЕЗ RUNTIME-БЕНЧМАРК під час побудови Dataset (лог 'Auto-choosing row-wise multi-threading, the overhead of testing was X seconds'). Це рішення залежить від виміряного wall-clock часу мікро-тесту в конкретний момент запуску -- тобто воно НЕ контролюється seed і теоретично може відрізнятись між двома прогонами на тій самій машині з тими самими бібліотеками, якщо момент запуску відрізняється (завантаження системи, кеш, throttling). Це -- поки що НЕ доведена, а лише правдоподібна кандидат-причина, яку варто тримати в полі зору при інтерпретації розділу 5 нижче.

❗ Версії бібліотек / кількість потоків / execution mode ОРИГІНАЛЬНОГО прогону Step 5

## 2. Незалежне перевідтворення UNSW feature contract і split (лише UNSW, той самий frozen код)

Той самий код, що й у Step 5D.2b (розділи 5.1-5.4), обмежений лише UNSW -- щоб отримати
train/val/test дані для forensic-експериментів нижче без залежності від in-memory стану
іншого notebook.

In [3]:
FROZEN_BOUNDARIES_UNSW = dict(
    file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
    label_col="attack_cat", benign_value="Normal",
    boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
    boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
)
FROZEN_SPLIT_SIZES_UNSW = dict(n_test=308_935, n_pos_test=25_354)

raw_unsw = pd.read_parquet(ENRICHED_DIR / FROZEN_BOUNDARIES_UNSW["file"])
print(f"UNSW: свіжо завантажено {len(raw_unsw):,} рядків, {len(raw_unsw.columns)} колонок")


UNSW: свіжо завантажено 2,059,418 рядків, 62 колонок


In [4]:
import re

IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)
TIME_COLS_UNSW = {"Stime", "Ltime"}
LABEL_COL_UNSW = "attack_cat"

def classify_column_unsw(col, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)"
    if col == LABEL_COL_UNSW:
        return "target"
    if col in TIME_COLS_UNSW:
        return "time"
    if IP_PATTERN.match(col):
        return "network identifier (IP)"
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)"
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata"
    if col.lower() == "label":
        return "target-derived (redundant)"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)"
    return "predictor"

INCLUDE_ROLES = {"predictor", "predictor (categorical)", "predictor (overlap risk)"}
roles_unsw = {col: classify_column_unsw(col, raw_unsw) for col in raw_unsw.columns}
f0_unsw = [c for c, r in roles_unsw.items() if r in INCLUDE_ROLES]
assert len(f0_unsw) == 41, f"UNSW F0 = {len(f0_unsw)}, очікувалось 41 (frozen Step 5D.1)"
print(f"✅ UNSW F0 = {len(f0_unsw)} predictors (frozen)")

F1_ADDITIONS_UNSW = [
    "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_epss_coverage",
    "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
]
f1_unsw = f0_unsw + F1_ADDITIONS_UNSW
assert len(f1_unsw) == 46, f"UNSW F1 = {len(f1_unsw)}, очікувалось 46"
FEATURE_SETS_UNSW = {"F0": f0_unsw, "F1": f1_unsw}
print(f"✅ UNSW F1 = {len(f1_unsw)} predictors (F0 + {len(F1_ADDITIONS_UNSW)} CTI)")


✅ UNSW F0 = 41 predictors (frozen)
✅ UNSW F1 = 46 predictors (F0 + 5 CTI)


In [5]:
df_unsw = raw_unsw.copy()
df_unsw["_is_attack"] = (df_unsw[FROZEN_BOUNDARIES_UNSW["label_col"]] != FROZEN_BOUNDARIES_UNSW["benign_value"]).astype(int)

if "ct_ftp_cmd" in df_unsw.columns:
    df_unsw["ct_ftp_cmd"] = pd.to_numeric(df_unsw["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

t_unsw = pd.to_datetime(df_unsw["Stime"], unit="s", errors="raise") \
    if pd.api.types.is_integer_dtype(df_unsw["Stime"]) else pd.to_datetime(df_unsw["Stime"])

train_mask = t_unsw < FROZEN_BOUNDARIES_UNSW["boundary_1"]
val_mask = (t_unsw >= FROZEN_BOUNDARIES_UNSW["boundary_1"]) & (t_unsw < FROZEN_BOUNDARIES_UNSW["boundary_2"])
test_mask = t_unsw >= FROZEN_BOUNDARIES_UNSW["boundary_2"]

train_df_unsw = df_unsw[train_mask].copy()
val_df_unsw = df_unsw[val_mask].copy()
test_df_unsw = df_unsw[test_mask].copy()

n_test, n_pos_test = len(test_df_unsw), int(test_df_unsw["_is_attack"].sum())
assert n_test == FROZEN_SPLIT_SIZES_UNSW["n_test"], f"n_test={n_test:,} != frozen {FROZEN_SPLIT_SIZES_UNSW['n_test']:,}"
assert n_pos_test == FROZEN_SPLIT_SIZES_UNSW["n_pos_test"], f"n_pos_test={n_pos_test:,} != frozen {FROZEN_SPLIT_SIZES_UNSW['n_pos_test']:,}"
print(f"✅ UNSW split точно збігається з frozen Step 5C.2 (n_test={n_test:,}, n_pos_test={n_pos_test:,})")

categorical_candidates_unsw = [c for c in FEATURE_SETS_UNSW["F1"]
                                if not pd.api.types.is_numeric_dtype(df_unsw[c]) or df_unsw[c].dtype == bool]
assert "ct_ftp_cmd" not in categorical_candidates_unsw
train_categories_unsw = {}
for col in categorical_candidates_unsw:
    cats = pd.Categorical(train_df_unsw[col]).categories
    train_categories_unsw[col] = cats
    train_df_unsw[col] = pd.Categorical(train_df_unsw[col], categories=cats)
    val_df_unsw[col] = pd.Categorical(val_df_unsw[col], categories=cats)
    test_df_unsw[col] = pd.Categorical(test_df_unsw[col], categories=cats)

print(f"\nКатегоріальні колонки (UNSW): {categorical_candidates_unsw}")


✅ UNSW split точно збігається з frozen Step 5C.2 (n_test=308,935, n_pos_test=25,354)

Категоріальні колонки (UNSW): ['proto', 'state', 'service', 'cve_ctx_service']


## 3. Categorical feature audit -- dtype, кількість категорій, точний code→category mapping

LightGBM кодує категоріальні ознаки внутрішньо цілими кодами (порядок `categories` визначає
код). Якщо оригінальний прогін Step 5D.2 присвоїв категоріям ІНШИЙ порядок (наприклад, через
іншу версію pandas з іншим порядком `unique()`/сортування), split-точки на цій ознаці будуть
закодовані по-іншому навіть при однакових даних і однаковому seed. Це друкується тут явно,
а не лише хешується (Step 5D.2b, розділ 7b), щоб можна було візуально звірити, якщо колись
зʼявиться історичний запис категорій з оригінального прогону.

In [6]:
cat_audit_rows = []
for col in categorical_candidates_unsw:
    cats = train_categories_unsw[col]
    print(f"\n{col}: dtype={train_df_unsw[col].dtype}  n_categories={len(cats)}")
    mapping = list(enumerate(cats))
    if len(mapping) <= 30:
        for code_, cat in mapping:
            print(f"    code {code_}: {cat!r}")
    else:
        for code_, cat in mapping[:15]:
            print(f"    code {code_}: {cat!r}")
        print(f"    ... ({len(mapping) - 30} категорій приховано) ...")
        for code_, cat in mapping[-15:]:
            print(f"    code {code_}: {cat!r}")
    cat_audit_rows.append(dict(
        column=col, dtype=str(train_df_unsw[col].dtype), n_categories=len(cats),
        categories="|".join(map(str, cats)),
    ))

pd.DataFrame(cat_audit_rows).to_csv(AUDIT_OUT_DIR / "unsw_categorical_contract.csv", index=False)
print(f"\n✅ Categorical contract збережено в {AUDIT_OUT_DIR / 'unsw_categorical_contract.csv'}")



proto: dtype=category  n_categories=135
    code 0: '3pc'
    code 1: 'a/n'
    code 2: 'aes-sp3-d'
    code 3: 'any'
    code 4: 'argus'
    code 5: 'aris'
    code 6: 'arp'
    code 7: 'ax.25'
    code 8: 'bbn-rcc'
    code 9: 'bna'
    code 10: 'br-sat-mon'
    code 11: 'cbt'
    code 12: 'cftp'
    code 13: 'chaos'
    code 14: 'compaq-peer'
    ... (105 категорій приховано) ...
    code 120: 'udp'
    code 121: 'udt'
    code 122: 'unas'
    code 123: 'uti'
    code 124: 'vines'
    code 125: 'visa'
    code 126: 'vmtp'
    code 127: 'vrrp'
    code 128: 'wb-expak'
    code 129: 'wb-mon'
    code 130: 'wsn'
    code 131: 'xnet'
    code 132: 'xns-idp'
    code 133: 'xtp'
    code 134: 'zero'

state: dtype=category  n_categories=16
    code 0: 'ACC'
    code 1: 'CLO'
    code 2: 'CON'
    code 3: 'ECO'
    code 4: 'ECR'
    code 5: 'FIN'
    code 6: 'INT'
    code 7: 'MAS'
    code 8: 'PAR'
    code 9: 'REQ'
    code 10: 'RST'
    code 11: 'TST'
    code 12: 'TXD'
    code 13: 'UR

## 4. LightGBM Dataset feature metadata -- явне підтвердження, які ознаки categorical

Підтверджує, що `lgb.Dataset` фактично отримав той самий список `categorical_feature`, який
очікується з розділу 2-3, а не default/auto-визначення LightGBM.

In [7]:
for feature_set_name in ["F0", "F1"]:
    features = FEATURE_SETS_UNSW[feature_set_name]
    cat_cols_fs = [c for c in features if train_df_unsw[c].dtype.name == "category"]
    print(f"UNSW-{feature_set_name}: {len(features)} ознак, з них categorical={cat_cols_fs}")


UNSW-F0: 41 ознак, з них categorical=['proto', 'state', 'service']
UNSW-F1: 46 ознак, з них categorical=['proto', 'state', 'service', 'cve_ctx_service']


## 5. Незалежне навчання з ПОВНОЮ validation AP trajectory (не лише фінальний test AP)

Той самий frozen `unweighted_average_precision` feval і ті самі `LGB_PARAMS`/`num_boost_round`/
`early_stopping_rounds`, що й у Step 5D.2b -- але тепер з `lgb.record_evaluation()`, який
зберігає AP на КОЖНІЙ ітерації валідації, а не лише в `best_iteration`. Це виконується для
ДВОХ конфігурацій (як і в Step 5D.2b, розділ 7d): `default` (frozen LGB_PARAMS без
модифікацій, той самий execution path, що дав 277/249) і `deterministic` (ЕКСПЕРИМЕНТАЛЬНА,
НЕ frozen, `deterministic=True, force_row_wise=True` + фіксовані seeds, що дав 267/344).

In [8]:
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

DETERMINISTIC_PARAM_OVERRIDES = dict(
    bagging_seed=RANDOM_STATE, feature_fraction_seed=RANDOM_STATE, data_random_seed=RANDOM_STATE,
    deterministic=True, force_row_wise=True,
)

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True

def train_with_trajectory(feature_set_name, param_overrides=None):
    features = FEATURE_SETS_UNSW[feature_set_name]
    X_train, y_train = train_df_unsw[features], train_df_unsw["_is_attack"]
    X_val, y_val = val_df_unsw[features], val_df_unsw["_is_attack"]
    X_test, y_test = test_df_unsw[features], test_df_unsw["_is_attack"]

    cat_cols = [c for c in features if X_train[c].dtype.name == "category"]
    n_pos_train, n_neg_train = int(y_train.sum()), len(y_train) - int(y_train.sum())
    params = dict(LGB_PARAMS, scale_pos_weight=n_neg_train / n_pos_train)
    if param_overrides:
        params.update(param_overrides)

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols if cat_cols else "auto")

    evals_result = {}
    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=0),
            lgb.record_evaluation(evals_result),
        ],
    )
    test_scores = model.predict(X_test, num_iteration=model.best_iteration)
    test_ap = average_precision_score(y_test, test_scores)
    return dict(
        best_iteration=model.best_iteration, test_ap=test_ap,
        val_ap_trajectory=evals_result["val"]["ap"],
    )


trajectories = {}
for feature_set_name in ["F0", "F1"]:
    for config_name, overrides in [("default", None), ("deterministic", DETERMINISTIC_PARAM_OVERRIDES)]:
        print(f"\n{'='*20} UNSW-{feature_set_name} [{config_name}] {'='*20}")
        result = train_with_trajectory(feature_set_name, overrides)
        trajectories[(feature_set_name, config_name)] = result
        bi, tap, tlen = result["best_iteration"], result["test_ap"], len(result["val_ap_trajectory"])
        print(f"best_iteration={bi}  test_ap={tap:.6f}  trajectory_length={tlen}")



==================== UNSW-F0 [default] ====================
[LightGBM] [Info] Number of positive: 49346, number of negative: 1392222
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.161994 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5715
[LightGBM] [Info] Number of data points in the train set: 1441568, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034231 -> initscore=-3.339800
[LightGBM] [Info] Start training from score -3.339800
best_iteration=277  test_ap=0.982353  trajectory_length=307

==================== UNSW-F0 [deterministic] ====================
[LightGBM] [Info] Number of positive: 49346, number of negative: 1392222
[LightGBM] [Info] Total Bins 5677
[LightGBM] [Info] Number of data points in the train set: 1441568, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=

## 6. Порівняння trajectory в області ранніх ітерацій (де оригінал обрав iteration=33 для F0)

Оригінальна (Step 5D.2) trajectory тут НЕ представлена -- вона ніколи не зберігалась. Це не
пропуск, а чесна межа цього аудиту (див. розділ 7 нижче). Показано лише, ЯК val AP
розвивається у `default` і `deterministic` конфігураціях поточного середовища, і де саме
`test_ap=0.973652` (оригінальне ЗНАЧЕННЯ, не ітерація) міг би теоретично відповідати точці
на цих кривих -- це орієнтир для читача, а НЕ доказ, оскільки val_ap ≠ test_ap.

In [9]:
ORIGINAL_UNSW_F0_TEST_AP = 0.973652
ORIGINAL_UNSW_F0_BEST_ITER = 33
ORIGINAL_UNSW_F1_TEST_AP = 0.983969
ORIGINAL_UNSW_F1_BEST_ITER = 290

for feature_set_name, orig_ap, orig_iter in [
    ("F0", ORIGINAL_UNSW_F0_TEST_AP, ORIGINAL_UNSW_F0_BEST_ITER),
    ("F1", ORIGINAL_UNSW_F1_TEST_AP, ORIGINAL_UNSW_F1_BEST_ITER),
]:
    print(f"\n{'='*78}\nUNSW-{feature_set_name}: val AP trajectory навколо оригінального "
          f"best_iteration={orig_iter} (оригінальний TEST AP={orig_ap:.6f} наведено лише як "
          f"орієнтир -- це не val AP і не доказ збігу)\n{'='*78}")
    window_lo, window_hi = max(1, orig_iter - 10), orig_iter + 15
    for config_name in ["default", "deterministic"]:
        entry = trajectories[(feature_set_name, config_name)]
        traj = entry["val_ap_trajectory"]
        bi = entry["best_iteration"]
        print(f"\n  [{config_name}]  best_iteration={bi}  trajectory_length={len(traj)}")
        for it in range(window_lo, min(window_hi, len(traj)) + 1):
            marker = "  <-- оригінальний best_iteration" if it == orig_iter else ""
            print(f"    iter {it:>4}: val_ap={traj[it - 1]:.6f}{marker}")



UNSW-F0: val AP trajectory навколо оригінального best_iteration=33 (оригінальний TEST AP=0.973652 наведено лише як орієнтир -- це не val AP і не доказ збігу)

  [default]  best_iteration=277  trajectory_length=307
    iter   23: val_ap=0.963724
    iter   24: val_ap=0.963622
    iter   25: val_ap=0.963972
    iter   26: val_ap=0.964187
    iter   27: val_ap=0.964300
    iter   28: val_ap=0.964442
    iter   29: val_ap=0.964458
    iter   30: val_ap=0.964446
    iter   31: val_ap=0.964440
    iter   32: val_ap=0.964516
    iter   33: val_ap=0.964458  <-- оригінальний best_iteration
    iter   34: val_ap=0.964533
    iter   35: val_ap=0.964398
    iter   36: val_ap=0.964407
    iter   37: val_ap=0.964354
    iter   38: val_ap=0.964355
    iter   39: val_ap=0.964266
    iter   40: val_ap=0.964337
    iter   41: val_ap=0.964239
    iter   42: val_ap=0.964190
    iter   43: val_ap=0.964002
    iter   44: val_ap=0.963972
    iter   45: val_ap=0.963714
    iter   46: val_ap=0.963006
    iter

## 7. Явна заява: оригінальна (Step 5D.2) validation AP trajectory НЕ відновлена

Оригінальний прогін Step 5D.2 зберіг лише агреговані метрики (`test_ap`, `best_iteration`
тощо) у `step5d2_f0_f1_results_h_flow.csv` -- НЕ `model` object і НЕ `evals_result_`/
per-iteration trajectory. Тому пряме порівняння "оригінальна trajectory vs поточна
trajectory" технічно неможливе з наявних артефактів. Це НЕ вигадується і не апроксимується
тут -- якщо оригінальний Jupyter kernel того прогону ще живий десь (малоймовірно) або
збережений `.ipynb` з виконаними output-комірками містить друк AP по ітераціях (наприклад,
verbose-лог LightGBM), це єдине можливе джерело відновлення. Якщо такого джерела немає --
питання "де саме розходяться траєкторії" залишається methodologically unresolved, і про це
треба прямо писати в methodology-розділі статті, а не мовчати.

In [10]:
print(
    "❌ Оригінальна (Step 5D.2) val AP trajectory: НЕ ВІДНОВЛЕНА -- відповідних артефактів "
    "(model object / evals_result_ / per-iteration лог) не існує в збережених файлах "
    "RESULTS_DIR. Порівняння в розділі 6 вище обмежене лише ПОТОЧНИМИ (default та "
    "deterministic) trajectory."
)


❌ Оригінальна (Step 5D.2) val AP trajectory: НЕ ВІДНОВЛЕНА -- відповідних артефактів (model object / evals_result_ / per-iteration лог) не існує в збережених файлах RESULTS_DIR. Порівняння в розділі 6 вище обмежене лише ПОТОЧНИМИ (default та deterministic) trajectory.


## 8. Інтерпретаційна гіпотеза: "Auto-choosing row/col-wise" -- runtime-heuristика, не контрольована seed

У логах LightGBM (Step 5D.2b) видно рядок:

```
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.12... seconds.
```

Це рішення (row-wise vs col-wise виконання гістограм) приймається LightGBM через runtime
мікро-бенчмарк під час побудови `Dataset` -- воно залежить від виміряного wall-clock часу в
конкретний момент запуску, а НЕ від `seed`. Теоретично воно може відрізнятись між двома
прогонами на одній і тій самій машині з тими самими бібліотеками й тими самими даними, якщо
момент запуску відрізняється (завантаження системи, I/O, кеш). Розділ 7d (Step 5D.2b) уже
показав, що примусове `force_row_wise=True` дає ІНШИЙ (267/344), а не оригінальний (33/290)
результат -- це узгоджується з тим, що причина НЕ зводиться просто до "row-wise vs col-wise",
але не виключає, що вона є ОДНІЄЮ з кількох взаємодіючих причин (разом із версією бібліотеки,
apparatним оточенням тощо). Це ПРАВДОПОДІБНА, АЛЕ НЕ ДОВЕДЕНА гіпотеза -- для остаточного
підтвердження знадобився б ізольований тест (`force_col_wise=True` окремо, без інших змін
детермінізму) і, в ідеалі, історичний запис оточення оригінального прогону.

## 9. Збереження форензик-артефактів

Повні trajectory (усі ітерації, обидві конфігурації) зберігаються у CSV для подальшого
аналізу поза цим notebook -- наприклад, для побудови графіка val AP curve.

In [11]:
for (feature_set_name, config_name), result in trajectories.items():
    traj_df = pd.DataFrame({
        "iteration": range(1, len(result["val_ap_trajectory"]) + 1),
        "val_ap": result["val_ap_trajectory"],
    })
    traj_df.to_csv(
        AUDIT_OUT_DIR / f"unsw_{feature_set_name}_{config_name}_val_ap_trajectory.csv",
        index=False,
    )

forensic_summary = dict(
    env_info=env_info,
    n_cpu=n_cpu,
    omp_num_threads=omp_threads,
    best_iterations={
        f"{fs}_{cfg}": trajectories[(fs, cfg)]["best_iteration"]
        for fs, cfg in trajectories
    },
    test_ap={
        f"{fs}_{cfg}": trajectories[(fs, cfg)]["test_ap"]
        for fs, cfg in trajectories
    },
    original_values=dict(
        UNSW_F0=dict(best_iteration=ORIGINAL_UNSW_F0_BEST_ITER, test_ap=ORIGINAL_UNSW_F0_TEST_AP),
        UNSW_F1=dict(best_iteration=ORIGINAL_UNSW_F1_BEST_ITER, test_ap=ORIGINAL_UNSW_F1_TEST_AP),
    ),
    original_trajectory_recovered=False,
    original_environment_recovered=False,
)
with open(AUDIT_OUT_DIR / "unsw_forensic_summary.json", "w") as f:
    json.dump(forensic_summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_forensic:
  unsw_F0_default_val_ap_trajectory.csv
  unsw_F0_deterministic_val_ap_trajectory.csv
  unsw_F1_default_val_ap_trajectory.csv
  unsw_F1_deterministic_val_ap_trajectory.csv
  unsw_categorical_contract.csv
  unsw_forensic_summary.json


## 10. Підсумок цього forensic-кроку

Заповнюється фактичним виводом після запуску:

- Чи розходяться `default` і `deterministic` trajectory для UNSW-F0 вже в області ранніх
  ітерацій (1-50), чи лише пізніше -- це вказує, чи проблема виникає під час побудови дерев
  з самого початку, чи накопичується поступово.
- Чи узгоджується categorical contract (розділ 3) із очікуваннями (немає несподіваних
  категорій чи порядку).
- Явна фіксація: оригінальна trajectory й оточення НЕ відновлені -- це залишається
  документованою межею цього аудиту, а не прихованою прогалиною.

Наступний крок (якщо буде виявлено історичний conda env/pip freeze/git-лог з часу
оригінального Step 5D.2 прогону) -- встановити версію LightGBM того часу і спробувати
відтворити С САМЕ нею, а не просто експериментувати з `deterministic=True` у поточній
версії.